# Job Description Skills Extractor
## Testing and Evaluation

This notebook loads the trained NER pipeline created in Notebook 2,
tests skill extraction on sample job descriptions, evaluates the
predictions, and prepares the model for deployment.

### Importing libraries

In [2]:
import spacy
import pandas as pd
import json
import re

from pathlib import Path
from spacy import displacy

#### Connecting Notebook 3 to the project

In [16]:
# Get the NLP_PROJECT folder
BASE_DIR = Path.cwd().parent

# Project folders
DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "models"

# Project files
CLEAN_DATA_FILE = DATA_DIR / "clean_jobs.csv"
NER_DATA_FILE = DATA_DIR / "ner_training_data.json"
MODEL_PATH = MODEL_DIR / "skill_ner_model"

print("Base directory:", BASE_DIR)
print("Data directory:", DATA_DIR)
print("Models directory:", MODEL_DIR)

Base directory: C:\Users\AFRIN\474 Classroom Practice\NLP_PROJECT
Data directory: C:\Users\AFRIN\474 Classroom Practice\NLP_PROJECT\data
Models directory: C:\Users\AFRIN\474 Classroom Practice\NLP_PROJECT\models


### Checking that Notebook 1 and Notebook 2 files exist

In [19]:
print("Checking project files...")
print("-" * 50)

print("clean_jobs.csv:",
      "FOUND" if CLEAN_DATA_FILE.exists() else "NOT FOUND")

print("ner_training_data.json:",
      "FOUND" if NER_DATA_FILE.exists() else "NOT FOUND")

print("skill_ner_model:",
      "FOUND" if MODEL_PATH.exists() else "NOT FOUND")

Checking project files...
--------------------------------------------------
clean_jobs.csv: FOUND
ner_training_data.json: FOUND
skill_ner_model: FOUND


### Loading the cleaned dataset from Notebook 1

In [18]:
clean_df = pd.read_csv(CLEAN_DATA_FILE)

print("Cleaned dataset loaded successfully!")
print("Dataset shape:", clean_df.shape)

Cleaned dataset loaded successfully!
Dataset shape: (28498, 12)


### Checking dataset columns

In [20]:
print("Dataset columns:")
print("-" * 50)

for column in clean_df.columns:
    print(column)

Dataset columns:
--------------------------------------------------
Uniq Id
Crawl Timestamp
Job Title
Job Salary
Job Experience Required
Key Skills
Role Category
Location
Functional Area
Industry
Role
clean_skills


### Displaying sample data

In [21]:
clean_df[["Job Title", "Key Skills"]].head(10)

,Job Title,Key Skills
0,Digital Media Planner,Media Planning| Digital Media
1,Online Bidding Executive,pre sales| closing| software knowledge| clien...
2,Trainee Research/ Research Executive- Hi- Tec...,Computer science| Fabrication| Quality check|...
3,Technical Support,Technical Support
4,Software Test Engineer -hyderabad,manual testing| test engineering| test cases|...
5,Opening For Adobe Analytics Specialist,adobe experience manager| digital| digital ma...
6,Sales- Fresher-for Leading Property Consultant,channel partners| real estate| negotiation| p...
7,Opportunity For Azure Devops Architect For Hy...,TFS| Azure| Git| VSTS| Docker| DynaTrace| Spl...
8,BDE- PUNE,Bde
9,Technical Support/ Product Support,technical support| support services| applicat...


### Load the NER training data from Notebook 2

In [22]:
with open(NER_DATA_FILE, "r", encoding="utf-8") as f:
    ner_training_data = json.load(f)

print("NER training data loaded successfully!")
print("Number of training examples:", len(ner_training_data))

NER training data loaded successfully!
Number of training examples: 28496


#### Inspecting one training example

In [23]:
print("First NER training example:")
print("-" * 50)

print(ner_training_data[0])

First NER training example:
--------------------------------------------------
{'text': 'media planning| digital media', 'entities': [[0, 14, 'SKILL'], [16, 29, 'SKILL']]}


### Loading the trained NER pipeline

In [24]:
nlp = spacy.load(MODEL_PATH)

print("NER pipeline loaded successfully!")
print("Pipeline components:", nlp.pipe_names)

NER pipeline loaded successfully!
Pipeline components: ['entity_ruler']


### Checking the EntityRuler

In [25]:
entity_ruler = nlp.get_pipe("entity_ruler")

print("EntityRuler loaded successfully!")
print("Number of patterns:", len(entity_ruler.patterns))

EntityRuler loaded successfully!
Number of patterns: 14629


### Checking available labels

In [27]:
labels = set()

for pattern in entity_ruler.patterns:
    labels.add(pattern["label"])

print("Available entity labels:")
print("-" * 50)

for label in sorted(labels):
    print(label)

Available entity labels:
--------------------------------------------------
SKILL


### Testing with one simple sentence

In [28]:
test_text = "We are looking for a Python Developer with SQL and AWS experience."

doc = nlp(test_text)

print("Input text:")
print(test_text)

print("\nExtracted entities:")
print("-" * 50)

for ent in doc.ents:
    print(ent.text, "→", ent.label_)

Input text:
We are looking for a Python Developer with SQL and AWS experience.

Extracted entities:
--------------------------------------------------
looking for → SKILL
Python Developer → SKILL
SQL → SKILL
AWS → SKILL


### Testing with several skills

In [29]:
test_text = """
We are looking for a Data Scientist with Python,
Pandas, NumPy, SQL, Machine Learning,
TensorFlow, Power BI and Docker experience.
"""

doc = nlp(test_text)

print("Input:")
print(test_text)

print("\nExtracted Skills:")
print("-" * 50)

for ent in doc.ents:
    if ent.label_ == "SKILL":
        print(ent.text)

Input:

We are looking for a Data Scientist with Python,
Pandas, NumPy, SQL, Machine Learning,
TensorFlow, Power BI and Docker experience.


Extracted Skills:
--------------------------------------------------
looking for
Data Scientist
Python
Pandas
SQL
Machine Learning
Power BI
Docker


### Creating the skill extraction function

In [30]:
def extract_skills(text, nlp_model):
    """
    Extract SKILL entities from a job description.
    """

    if pd.isna(text):
        return []

    text = str(text)

    doc = nlp_model(text)

    skills = []

    for ent in doc.ents:
        if ent.label_ == "SKILL":
            skills.append(ent.text.strip())

    # Remove duplicates while preserving order
    skills = list(dict.fromkeys(skills))

    return skills

### Testing the function

In [31]:
text = """
Looking for a Python Developer with Django,
SQL, AWS, Docker and Machine Learning.
"""

skills = extract_skills(text, nlp)

print("Extracted Skills:")
print("-" * 50)

for skill in skills:
    print("•", skill)

Extracted Skills:
--------------------------------------------------
• Looking for
• Python Developer
• Django
• SQL
• AWS
• Docker
• Machine Learning


### Creating expected-skill function

In [32]:
def get_expected_skills(text):
    """
    Convert the dataset's Key Skills field into a list of skills.
    """

    if pd.isna(text):
        return []

    text = str(text)

    # Split skills using common separators
    skills = re.split(r"[|,;/]", text)

    skills = [
        skill.strip()
        for skill in skills
        if skill.strip()
    ]

    return list(dict.fromkeys(skills))

### Testing expected skills

In [33]:
example_skills = clean_df["Key Skills"].iloc[0]

print("Original Key Skills:")
print(example_skills)

print("\nConverted skill list:")
print(get_expected_skills(example_skills))

Original Key Skills:
 Media Planning| Digital Media

Converted skill list:
['Media Planning', 'Digital Media']


### Creating job text

In [34]:
def create_job_text(row):
    title = str(row["Job Title"]) if pd.notna(row["Job Title"]) else ""
    skills = str(row["Key Skills"]) if pd.notna(row["Key Skills"]) else ""

    return title + " " + skills

### Creating combined text

In [35]:
clean_df["job_text"] = clean_df.apply(create_job_text, axis=1)

print("Combined job text created successfully!")

clean_df[["Job Title", "Key Skills", "job_text"]].head()

Combined job text created successfully!


,Job Title,Key Skills,job_text
0,Digital Media Planner,Media Planning| Digital Media,Digital Media Planner Media Planning| Digita...
1,Online Bidding Executive,pre sales| closing| software knowledge| clien...,Online Bidding Executive pre sales| closing|...
2,Trainee Research/ Research Executive- Hi- Tec...,Computer science| Fabrication| Quality check|...,Trainee Research/ Research Executive- Hi- Tec...
3,Technical Support,Technical Support,Technical Support Technical Support
4,Software Test Engineer -hyderabad,manual testing| test engineering| test cases|...,Software Test Engineer -hyderabad manual tes...


### Extracting skills from the complete dataset

In [36]:
texts = clean_df["job_text"].astype(str).tolist()

predicted_skills = []

print("Starting skill extraction...")
print("Total records:", len(texts))

for doc in nlp.pipe(texts, batch_size=100):

    skills = [
        ent.text.strip()
        for ent in doc.ents
        if ent.label_ == "SKILL"
    ]

    # Remove duplicates
    skills = list(dict.fromkeys(skills))

    predicted_skills.append(skills)

print("Skill extraction completed!")

Starting skill extraction...
Total records: 28498
Skill extraction completed!


### Adding predictions to dataframe

In [37]:
clean_df["predicted_skills"] = predicted_skills

print("Predicted skills added to dataset.")

Predicted skills added to dataset.


### Viewing results

In [38]:
clean_df[
    ["Job Title", "Key Skills", "predicted_skills"]
].head(20)

,Job Title,Key Skills,predicted_skills
0,Digital Media Planner,Media Planning| Digital Media,"[Digital Media Planner, Media, Digital Media]"
1,Online Bidding Executive,pre sales| closing| software knowledge| clien...,"[Online Bidding, Executive, pre, software, onl..."
2,Trainee Research/ Research Executive- Hi- Tec...,Computer science| Fabrication| Quality check|...,"[Trainee, Research, Tech, Operations, Computer..."
3,Technical Support,Technical Support,[Technical Support]
4,Software Test Engineer -hyderabad,manual testing| test engineering| test cases|...,"[Software Test Engineer, manual, test, web, we..."
5,Opening For Adobe Analytics Specialist,adobe experience manager| digital| digital ma...,"[Adobe Analytics, Specialist, adobe, digital, ..."
6,Sales- Fresher-for Leading Property Consultant,channel partners| real estate| negotiation| p...,"[Fresher, Leading, Property Consultant, channe..."
7,Opportunity For Azure Devops Architect For Hy...,TFS| Azure| Git| VSTS| Docker| DynaTrace| Spl...,"[Opportunity, Azure, Devops, Architect, Hydera..."
8,BDE- PUNE,Bde,"[PUNE, Bde]"
9,Technical Support/ Product Support,technical support| support services| applicat...,"[Technical, Product Support, technical, suppor..."


### Normalizing skills for evaluation

In [40]:
def normalize_skill(skill):
    return re.sub(r"\s+", " ", str(skill).strip().lower())

### Calculate Precision, Recall and F1

In [41]:
def calculate_metrics(expected, predicted):

    expected = {
        normalize_skill(skill)
        for skill in expected
        if skill
    }

    predicted = {
        normalize_skill(skill)
        for skill in predicted
        if skill
    }

    true_positive = len(expected & predicted)

    false_positive = len(predicted - expected)

    false_negative = len(expected - predicted)

    precision = (
        true_positive / (true_positive + false_positive)
        if (true_positive + false_positive) > 0
        else 0
    )

    recall = (
        true_positive / (true_positive + false_negative)
        if (true_positive + false_negative) > 0
        else 0
    )

    f1_score = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    return precision, recall, f1_score

### Evaluating the complete dataset

In [42]:
total_true_positive = 0
total_false_positive = 0
total_false_negative = 0

for _, row in clean_df.iterrows():

    expected = {
        normalize_skill(skill)
        for skill in get_expected_skills(row["Key Skills"])
    }

    predicted = {
        normalize_skill(skill)
        for skill in row["predicted_skills"]
    }

    total_true_positive += len(expected & predicted)
    total_false_positive += len(predicted - expected)
    total_false_negative += len(expected - predicted)

### Calculating final metrics

In [43]:
precision = (
    total_true_positive /
    (total_true_positive + total_false_positive)
    if (total_true_positive + total_false_positive) > 0
    else 0
)

recall = (
    total_true_positive /
    (total_true_positive + total_false_negative)
    if (total_true_positive + total_false_negative) > 0
    else 0
)

f1_score = (
    2 * precision * recall /
    (precision + recall)
    if (precision + recall) > 0
    else 0
)

print("NER SKILL EXTRACTION EVALUATION")
print("=" * 50)

print(f"True Positives : {total_true_positive}")
print(f"False Positives: {total_false_positive}")
print(f"False Negatives: {total_false_negative}")

print("-" * 50)

print(f"Precision: {precision:.4f} ({precision * 100:.2f}%)")
print(f"Recall   : {recall:.4f} ({recall * 100:.2f}%)")
print(f"F1 Score : {f1_score:.4f} ({f1_score * 100:.2f}%)")

NER SKILL EXTRACTION EVALUATION
True Positives : 42468
False Positives: 118509
False Negatives: 180517
--------------------------------------------------
Precision: 0.2638 (26.38%)
Recall   : 0.1905 (19.05%)
F1 Score : 0.2212 (22.12%)


### Finding the best matching examples

In [44]:
clean_df["expected_skills"] = clean_df["Key Skills"].apply(
    get_expected_skills
)

clean_df["expected_count"] = clean_df["expected_skills"].apply(len)

clean_df["predicted_count"] = clean_df["predicted_skills"].apply(len)

clean_df[
    [
        "Job Title",
        "expected_skills",
        "predicted_skills",
        "expected_count",
        "predicted_count"
    ]
].head(20)

,Job Title,expected_skills,predicted_skills,expected_count,predicted_count
0,Digital Media Planner,"[Media Planning, Digital Media]","[Digital Media Planner, Media, Digital Media]",2,3
1,Online Bidding Executive,"[pre sales, closing, software knowledge, clien...","[Online Bidding, Executive, pre, software, onl...",10,6
2,Trainee Research/ Research Executive- Hi- Tec...,"[Computer science, Fabrication, Quality check,...","[Trainee, Research, Tech, Operations, Computer...",10,9
3,Technical Support,[Technical Support],[Technical Support],1,1
4,Software Test Engineer -hyderabad,"[manual testing, test engineering, test cases,...","[Software Test Engineer, manual, test, web, we...",5,5
5,Opening For Adobe Analytics Specialist,"[adobe experience manager, digital, digital ma...","[Adobe Analytics, Specialist, adobe, digital, ...",8,6
6,Sales- Fresher-for Leading Property Consultant,"[channel partners, real estate, negotiation, p...","[Fresher, Leading, Property Consultant, channe...",5,6
7,Opportunity For Azure Devops Architect For Hy...,"[TFS, Azure, Git, VSTS, Docker, DynaTrace, Spl...","[Opportunity, Azure, Devops, Architect, Hydera...",8,6
8,BDE- PUNE,[Bde],"[PUNE, Bde]",1,2
9,Technical Support/ Product Support,"[technical support, support services, applicat...","[Technical, Product Support, technical, suppor...",7,8


### Finding rows where extraction differs

In [45]:
mismatch_df = clean_df[
    clean_df["expected_count"] != clean_df["predicted_count"]
]

print("Number of rows with different skill counts:",
      len(mismatch_df))

Number of rows with different skill counts: 25173


In [46]:
mismatch_df[
    [
        "Job Title",
        "expected_skills",
        "predicted_skills"
    ]
].head(20)

,Job Title,expected_skills,predicted_skills
0,Digital Media Planner,"[Media Planning, Digital Media]","[Digital Media Planner, Media, Digital Media]"
1,Online Bidding Executive,"[pre sales, closing, software knowledge, clien...","[Online Bidding, Executive, pre, software, onl..."
2,Trainee Research/ Research Executive- Hi- Tec...,"[Computer science, Fabrication, Quality check,...","[Trainee, Research, Tech, Operations, Computer..."
5,Opening For Adobe Analytics Specialist,"[adobe experience manager, digital, digital ma...","[Adobe Analytics, Specialist, adobe, digital, ..."
6,Sales- Fresher-for Leading Property Consultant,"[channel partners, real estate, negotiation, p...","[Fresher, Leading, Property Consultant, channe..."
7,Opportunity For Azure Devops Architect For Hy...,"[TFS, Azure, Git, VSTS, Docker, DynaTrace, Spl...","[Opportunity, Azure, Devops, Architect, Hydera..."
8,BDE- PUNE,[Bde],"[PUNE, Bde]"
9,Technical Support/ Product Support,"[technical support, support services, applicat...","[Technical, Product Support, technical, suppor..."
11,SEO Executive,"[website, web analytics, xml, link building, g...","[SEO Executive, web, google, case, maintaining]"
12,Workflow Coordinator,"[operations, workflow, tat, monitoring, mts, e...","[Workflow Coordinator, email]"


### Saving prediction results

In [47]:
prediction_file = DATA_DIR / "ner_prediction_results.csv"

output_df = clean_df[
    [
        "Job Title",
        "Key Skills",
        "predicted_skills"
    ]
].copy()

output_df["predicted_skills"] = output_df["predicted_skills"].apply(
    lambda x: ", ".join(x)
)

output_df.to_csv(
    prediction_file,
    index=False,
    encoding="utf-8"
)

print("Prediction results saved successfully!")
print("File:", prediction_file)

Prediction results saved successfully!
File: C:\Users\AFRIN\474 Classroom Practice\NLP_PROJECT\data\ner_prediction_results.csv


### Creating evaluation results JSON

In [48]:
evaluation_results = {
    "project": "Job Description Skills Extractor",
    "model_type": "spaCy EntityRuler",
    "entity_type": "SKILL",
    "dataset_records": int(len(clean_df)),
    "true_positive": int(total_true_positive),
    "false_positive": int(total_false_positive),
    "false_negative": int(total_false_negative),
    "precision": round(float(precision), 4),
    "recall": round(float(recall), 4),
    "f1_score": round(float(f1_score), 4)
}

evaluation_file = DATA_DIR / "evaluation_results.json"

with open(
    evaluation_file,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        evaluation_results,
        f,
        indent=4
    )

print("Evaluation results saved successfully!")
print("File:", evaluation_file)

Evaluation results saved successfully!
File: C:\Users\AFRIN\474 Classroom Practice\NLP_PROJECT\data\evaluation_results.json


### Displaying final evaluation

In [49]:
print("=" * 60)
print("FINAL NER EVALUATION")
print("=" * 60)

print(f"Dataset Records : {len(clean_df)}")
print(f"Precision       : {precision * 100:.2f}%")
print(f"Recall          : {recall * 100:.2f}%")
print(f"F1 Score        : {f1_score * 100:.2f}%")

print("=" * 60)

FINAL NER EVALUATION
Dataset Records : 28498
Precision       : 26.38%
Recall          : 19.05%
F1 Score        : 22.12%


### Final manual testing

In [50]:
test_cases = [
    "Python developer with Django and REST API experience.",
    
    "Data Scientist skilled in Python, Pandas, NumPy, SQL and Machine Learning.",
    
    "Frontend developer with HTML, CSS, JavaScript and React.",
    
    "Cloud engineer with AWS, Docker and Kubernetes.",
    
    "Machine Learning Engineer with TensorFlow, Python and Scikit Learn."
]

print("MANUAL NER TESTING")
print("=" * 60)

for i, text in enumerate(test_cases, 1):

    skills = extract_skills(text, nlp)

    print(f"\nTest {i}")
    print("Input:", text)
    print("Skills:", skills)

MANUAL NER TESTING

Test 1
Input: Python developer with Django and REST API experience.
Skills: ['Python developer', 'Django', 'REST API']

Test 2
Input: Data Scientist skilled in Python, Pandas, NumPy, SQL and Machine Learning.
Skills: ['Data Scientist', 'Python', 'Pandas', 'SQL', 'Machine Learning']

Test 3
Input: Frontend developer with HTML, CSS, JavaScript and React.
Skills: ['Frontend', 'HTML', 'CSS', 'JavaScript', 'React']

Test 4
Input: Cloud engineer with AWS, Docker and Kubernetes.
Skills: ['Cloud', 'engineer', 'AWS', 'Docker', 'Kubernetes']

Test 5
Input: Machine Learning Engineer with TensorFlow, Python and Scikit Learn.
Skills: ['Machine Learning', 'Engineer', 'Python']


### Final model summary

In [51]:
print("=" * 60)
print("MODEL SUMMARY")
print("=" * 60)

print("Model path:", MODEL_PATH)
print("Pipeline:", nlp.pipe_names)
print("Entity labels:", sorted(labels))
print("Number of EntityRuler patterns:", len(entity_ruler.patterns))
print("Dataset records:", len(clean_df))

print("\nEvaluation:")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1 Score:  {f1_score * 100:.2f}%")

MODEL SUMMARY
Model path: C:\Users\AFRIN\474 Classroom Practice\NLP_PROJECT\models\skill_ner_model
Pipeline: ['entity_ruler']
Entity labels: ['SKILL']
Number of EntityRuler patterns: 14629
Dataset records: 28498

Evaluation:
Precision: 26.38%
Recall:    19.05%
F1 Score:  22.12%


### Checking generated files

In [52]:
print("=" * 60)
print("PROJECT OUTPUT FILES")
print("=" * 60)

files_to_check = [
    CLEAN_DATA_FILE,
    NER_DATA_FILE,
    MODEL_PATH,
    DATA_DIR / "ner_prediction_results.csv",
    DATA_DIR / "evaluation_results.json"
]

for file in files_to_check:

    status = "FOUND" if file.exists() else "NOT FOUND"

    print(f"{status:10} : {file}")

PROJECT OUTPUT FILES
FOUND      : C:\Users\AFRIN\474 Classroom Practice\NLP_PROJECT\data\clean_jobs.csv
FOUND      : C:\Users\AFRIN\474 Classroom Practice\NLP_PROJECT\data\ner_training_data.json
FOUND      : C:\Users\AFRIN\474 Classroom Practice\NLP_PROJECT\models\skill_ner_model
FOUND      : C:\Users\AFRIN\474 Classroom Practice\NLP_PROJECT\data\ner_prediction_results.csv
FOUND      : C:\Users\AFRIN\474 Classroom Practice\NLP_PROJECT\data\evaluation_results.json
